# Zoom In：神经网络电路入门复现

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jonny-English/circuits-zoom-in/blob/main/notebooks/circuits_zoom_in_zh.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.8+](https://img.shields.io/badge/python-3.8+-blue.svg)](https://www.python.org/downloads/)

> **[English version](circuits_zoom_in_en.ipynb)** | 本 Notebook 复现 Olah et al. 2020《Zoom In: An Introduction to Circuits》的核心思想。

**目录**
- [第 0 节：导入库](#第-0-节导入库)
- [第 1 节：加载模型](#第-1-节加载模型)
- [第 2 节：特征可视化](#第-2-节特征可视化激活最大化)
- [第 3 节：真实图片验证](#第-3-节用真实图片验证)
- [第 4 节：方向调谐实验](#第-4-节方向调谐实验)
- [第 5 节：电路分析](#第-5-节电路分析曲线是如何被计算出来的)
- [第 6 节：普遍性验证](#第-6-节普遍性验证)
- [第 7 节：局限性与开放问题](#第-7-节局限性与开放问题)

---

文章提出三个核心主张：

| 主张 | 含义 |
|------|------|
| **特征** | 神经元检测人类可理解的视觉特征（曲线、纹理、颜色…） |
| **电路** | 特征之间通过权重连接，形成执行特定计算的电路 |
| **普遍性** | 不同模型、不同任务中，相同的特征和电路会反复出现 |

使用的模型：**InceptionV1（Inception5h）**，与原论文一致。

---
### 模型层结构速览

```
输入图片 (224×224 像素)
  conv2d0 → conv2d1 → conv2d2      ← 早期卷积层，检测颜色/边缘等简单特征
  mixed3a → mixed3b                ← 中浅层，开始出现曲线等中级特征
  mixed4a → mixed4b → ... → mixed4e  ← 中层，纹理/物体部件
  mixed5a → mixed5b                ← 深层，复杂语义特征
  softmax 输出 1000 个类别的概率
```

每个 `mixed` 块是一个 **Inception 模块**，内部有四条并行分支（1×1、3×3、5×5 卷积 + 池化），
最后把四条分支的输出在通道维度**拼接**起来。

## 第 0 节：导入库

In [ ]:
# Google Colab 用户请取消注释下面这行：
# !pip install torch torchvision torch-lucent matplotlib numpy Pillow

import torch
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
from PIL import Image, ImageDraw
import math, warnings, os
warnings.filterwarnings('ignore')

# ── CI 模式：在 GitHub Actions 中用极少步数快速验证代码可运行 ──
CI_MODE = os.environ.get('CI') == 'true'
默认步数 = 8 if CI_MODE else 512

# ── 中文字体配置 ──
def _配置中文字体():
    for path in ['/System/Library/Fonts/STHeiti Medium.ttc',
                 '/System/Library/Fonts/PingFang.ttc',
                 '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
                 '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
                 '/usr/share/fonts/wqy-zenhei/wqy-zenhei.ttc',
                 'C:/Windows/Fonts/msyh.ttc']:
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            name = fm.FontProperties(fname=path).get_name()
            matplotlib.rcParams['font.sans-serif'] = [name, 'DejaVu Sans']
            return name
    return None

_cn_font_name = _配置中文字体()
matplotlib.rcParams['axes.unicode_minus'] = False

from lucent.modelzoo import inceptionv1
from lucent.optvis import render, objectives

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device} | PyTorch: {torch.__version__}')
print(f'中文字体: {_cn_font_name or "未找到，中文可能显示为方框"}')
if CI_MODE:
    print('⚡ CI 模式：使用极少优化步数，仅验证代码可运行')


def 显示图片网格(图片列表, 标题列表, 总标题=None, 列数=4, 格子尺寸=3.2):
    """在网格中显示一组图片。"""
    行数 = math.ceil(len(图片列表) / 列数)
    图, 子图 = plt.subplots(行数, 列数, figsize=(列数*格子尺寸, 行数*格子尺寸))
    子图 = np.array(子图).flatten()
    for i, (图片, 标题) in enumerate(zip(图片列表, 标题列表)):
        子图[i].imshow(图片)
        子图[i].axis('off')
        子图[i].set_title(标题, fontsize=8.5)
    for j in range(len(图片列表), len(子图)):
        子图[j].axis('off')
    if 总标题:
        plt.suptitle(总标题, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

## 第 1 节：加载模型

In [ ]:
# 加载预训练的 InceptionV1 模型
# pretrained=True 表示使用在 ImageNet 上训练好的权重
# 首次运行会从网上下载约 27MB 的权重文件，之后会缓存到本地
model = inceptionv1(pretrained=True).to(device).eval()
# .eval() 切换到推理模式，关闭 Dropout 等训练专用层
print('模型加载完成')

# 用一张随机噪声图片测试模型是否正常工作
# torch.no_grad() 表示不计算梯度，节省内存
with torch.no_grad():
    test_input = torch.randn(1, 3, 224, 224, device=device)  # 1张、3通道(RGB)、224×224
    test_output = model(test_input)
print(f'输出形状: {test_output.shape}  ← 1张图片，1000个类别的分数')

In [ ]:
# 用「钩子（Hook）」查看各层的输出尺寸
# Hook 是 PyTorch 的一个功能：在某一层计算完之后，自动调用我们指定的函数
# 这里我们用它来记录每层的输出形状

目标层 = ['conv2d0', 'conv2d1', 'conv2d2',
           'mixed3a', 'mixed3b', 'mixed4a', 'mixed4e', 'mixed5b']

形状记录 = {}
钩子列表 = []

# 给每个目标层注册一个钩子
for 层名, 模块 in model.named_modules():
    if 层名 in 目标层:
        # n=层名 是为了避免 Python 闭包陷阱（捕获变量而非值）
        def _记录形状(m, 输入, 输出, n=层名):
            形状记录[n] = tuple(输出.shape)
        钩子列表.append(模块.register_forward_hook(_记录形状))

# 跑一次前向传播触发所有钩子
with torch.no_grad():
    model(torch.randn(1, 3, 224, 224, device=device))

# 用完后移除钩子，避免影响后续计算
for 钩子 in 钩子列表:
    钩子.remove()

print('层名称 → 输出形状 (batch大小, 通道数, 高度, 宽度)')
print('（通道数越多 = 能表达的特征越多；越深的层 H/W 越小但通道越多）')
print()
for 名称 in 目标层:
    形状 = 形状记录[名称]
    print(f'  {名称:12s}: {形状}')

---
## 第 2 节：特征可视化（激活最大化）

**问题**：一个神经元到底在「看」什么？

**方法**：从随机噪声图片开始，用梯度上升不断修改图片，
让目标神经元的激活值尽可能大，最终得到的图片就是该神经元的「偏好刺激」。

```
初始：随机噪声图片
  ↓ 反复迭代：向让神经元「更兴奋」的方向修改图片
结果：神经元最想看到的图案
```

lucent 在此基础上加了两个技巧让结果更清晰：
- **傅里叶参数化**：在频域优化，避免高频噪声
- **随机变换增强**：每步随机缩放/旋转图片，让特征更稳健

In [ ]:
def 可视化神经元(层名, 通道, 步数=默认步数, 显示=True, 标题=None):
    """
    对指定层的指定通道做特征可视化（激活最大化）。
    步数越多越清晰，越慢。CPU 上 512 步约 30 秒。
    返回 numpy 数组 (128, 128, 3)，像素值 [0,1]
    """
    目标 = objectives.channel(层名, 通道)
    结果列表 = render.render_vis(
        model, 目标,
        thresholds=(步数,),
        show_inline=False,
        progress=False,
    )
    图片 = 结果列表[0][0]
    if 显示:
        plt.figure(figsize=(3.5, 3.5))
        plt.imshow(图片)
        plt.axis('off')
        plt.title(标题 or f'{层名}:{通道}', fontsize=10)
        plt.tight_layout()
        plt.show()
    return 图片

print('正在生成 mixed3b:379 的特征可视化图（约 30 秒）...')
_ = 可视化神经元('mixed3b', 379, 步数=默认步数, 标题='mixed3b:379 曲线探测器')

In [ ]:
# 批量生成 8 个有代表性的神经元的可视化图
#
# mixed3b 共 480 通道，四条分支按顺序拼接：
#   通道   0 ~ 127  <- 1x1 分支
#   通道 128 ~ 319  <- 3x3 分支
#   通道 320 ~ 415  <- 5x5 分支
#   通道 416 ~ 479  <- 池化分支

代表性神经元 = [
    # (层名,       通道号,  中文描述)
    ('conv2d0',    0,    '早期：颜色/边缘'),
    ('conv2d1',    5,    '早期：颜色对比'),
    ('mixed3a',    0,    '中层：边缘探测器'),
    ('mixed3b',    6,    '高低频探测器'),
    ('mixed3b',  379,    '曲线探测器'),
    ('mixed3b',  425,    '池化分支特征'),
    ('mixed4a',   22,    '中层：纹理'),
    ('mixed4e',  254,    '深层：复杂特征'),
]

print(f'共 {len(代表性神经元)} 个神经元，CPU 约需 5~10 分钟')
print()

缓存 = {}
for 层名, 通道号, 描述 in 代表性神经元:
    print(f'  正在处理 {层名}:{通道号} ({描述})', end=' ... ', flush=True)
    图片 = 可视化神经元(层名, 通道号, 步数=默认步数, 显示=False)
    缓存[(层名, 通道号)] = (图片, 描述)
    print('完成')

print('\n全部完成！')

In [ ]:
显示图片网格(
    [缓存[(层, 通道)][0] for 层, 通道, _ in 代表性神经元],
    [f'{层}:{通道}\n{描述}' for 层, 通道, 描述 in 代表性神经元],
    总标题='特征可视化（激活最大化）\n浅层 -> 深层：简单特征 -> 复杂特征',
)

print()
print('观察要点：')
print('  conv2d0/1  -> 条纹、颜色渐变等最基础的视觉元素')
print('  mixed3a/3b -> 开始出现有方向性的曲线或纹理')
print('  mixed4a/4e -> 更复杂的重复纹理，甚至像材质或物体部件')

---
## 第 3 节：用真实图片验证

上一节是「合成图」——从噪声优化出来的最优刺激。
这一节换一个角度：在真实图片数据集里，**哪些图片让这个神经元激活最强？**

如果合成图和真实图在视觉上相似 → 说明这个神经元确实在检测真实世界里的某种视觉模式，
而不是对优化技巧的过拟合。

In [ ]:
from torch.utils.data import DataLoader

# 使用 CIFAR-10 测试集作为真实图片来源
# CIFAR-10：10类物体（飞机/汽车/鸟/猫等），原始尺寸 32×32，约 30MB
# 缺点：分辨率较低，放大到 224×224 后比较模糊，但足够演示概念
图片变换 = T.Compose([
    T.Resize(224),                                          # 放大到 224×224
    T.ToTensor(),                                           # 转为 PyTorch 张量
    T.Normalize(mean=[0.485, 0.456, 0.406],                # 归一化（ImageNet 均值/标准差）
                std=[0.229, 0.224, 0.225]),                 # 让模型输入分布与训练时一致
])

数据集 = torchvision.datasets.CIFAR10(
    os.path.join(os.path.dirname(os.getcwd()), 'data', 'cifar10'),       # 下载到临时目录
    train=False,          # 用测试集（不参与训练）
    download=True,        # 如果没有则自动下载
    transform=图片变换
)
数据加载器 = DataLoader(数据集, batch_size=64, shuffle=False, num_workers=0)
print(f'CIFAR-10 测试集加载完成，共 {len(数据集)} 张图片')

In [ ]:
def 反归一化(张量):
    """把归一化后的张量还原成可显示的图片。
    输入: tensor 形状 (3, H, W)
    输出: numpy 数组 形状 (H, W, 3)，像素值范围 [0, 1]
    """
    均值 = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    标准差 = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    还原后 = (张量.cpu() * 标准差 + 均值).clamp(0, 1)  # clamp 确保像素值在 [0,1]
    return 还原后.permute(1, 2, 0).numpy()             # (C,H,W) → (H,W,C)


def 找最高激活图片(模型, 数据加载器, 层名, 通道, 取前K=9, 最多批次=10):
    """在数据集里找出让指定神经元激活最高的图片。

    做法：对每批图片前向传播，用 Hook 记录目标通道的激活值，
    最后取激活值最高的前 K 张。
    """
    激活值列表 = []
    图片列表 = []
    暂存 = {}

    # Hook：每次目标层计算完后，记录当前 batch 里每张图的激活值
    # o 的形状是 (batch, 通道数, H, W)
    # 对 H×W 取均值，得到每张图在该通道上的「代表性激活强度」
    def 记录激活(m, 输入, 输出):
        暂存['激活'] = 输出[:, 通道].mean((1, 2)).detach().cpu()

    钩子 = dict(模型.named_modules())[层名].register_forward_hook(记录激活)

    with torch.no_grad():
        for 批次序号, (图片批, _) in enumerate(数据加载器):
            if 批次序号 >= 最多批次:
                break
            模型(图片批.to(device))
            激活值列表.append(暂存['激活'])
            图片列表.append(图片批)

    钩子.remove()  # 用完必须移除，否则会一直占用内存

    所有激活 = torch.cat(激活值列表)   # (总图片数,)
    所有图片 = torch.cat(图片列表)     # (总图片数, 3, 224, 224)

    # 找出激活值最大的前 K 张
    最高索引 = 所有激活.topk(取前K).indices
    return 所有图片[最高索引], 所有激活[最高索引]


# 对 mixed3b:379（曲线探测器）做搜索
目标层名 = 'mixed3b'
目标通道 = 379
print(f'在 CIFAR-10 里搜索最能激活 {目标层名}:{目标通道} 的图片...')
最高图片, 最高激活值 = 找最高激活图片(model, 数据加载器, 目标层名, 目标通道)
print('搜索完成')

In [ ]:
合成图, _ = 缓存[(目标层名, 目标通道)]

图 = plt.figure(figsize=(14, 5.5))
网格 = gridspec.GridSpec(3, 7, figure=图, wspace=0.08, hspace=0.08)

左图 = 图.add_subplot(网格[:, :2])
左图.imshow(合成图)
左图.axis('off')
左图.set_title(f'特征可视化\n（激活最大化）\n{目标层名}:{目标通道}', fontsize=10)

中图 = 图.add_subplot(网格[:, 2])
中图.text(0.5, 0.5, 'vs', ha='center', va='center', fontsize=20, color='gray', transform=中图.transAxes)
中图.axis('off')

for i in range(9):
    行, 列 = divmod(i, 3)
    子图 = 图.add_subplot(网格[行, 列+3])
    子图.imshow(反归一化(最高图片[i]))
    子图.axis('off')
    子图.set_title(f'激活:{最高激活值[i]:.2f}', fontsize=7)

图.text(0.67, 1.0, '激活最高的 9 张真实图片（CIFAR-10）', ha='center', fontsize=11)
plt.show()

print()
print('如何解读：')
print('  左边的合成图 = 该神经元「理想中」想看到的图案')
print('  右边 = 真实图片里让它最兴奋的那些')
print('  两边视觉相似 -> 神经元在检测真实世界的某种特征')
print('  注：这是定性比较。定量方法（如余弦相似度）可进一步验证，留待未来工作。')

---
## 第 4 节：方向调谐实验

原论文展示：曲线探测器对**特定方向**的曲线响应最强，转动方向后激活值会下降。
这与神经科学中人类视觉皮层 V1 区简单细胞的「方向选择性」高度一致。

**实验方法**：生成不同方向的弧线，分别测量激活值，绘制极坐标图。
极坐标图上突出的「尖峰」方向 = 该神经元最偏好的曲线方向。

In [ ]:
def 生成弧线图片(角度, 图片尺寸=224, 半径=60, 线宽=5):
    """在灰色背景上画一条白色弧线，返回归一化 tensor (1,3,224,224)"""
    图片 = Image.new('RGB', (图片尺寸, 图片尺寸), (128, 128, 128))
    画笔 = ImageDraw.Draw(图片)
    cx, cy = 图片尺寸 // 2, 图片尺寸 // 2
    外框 = [cx-半径, cy-半径, cx+半径, cy+半径]
    画笔.arc(外框, start=角度-60, end=角度+60, fill=(255, 255, 255), width=线宽)
    变换 = T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return 变换(图片).unsqueeze(0)

图, 子图数组 = plt.subplots(1, 6, figsize=(12, 2.2))
for 子图, 角度 in zip(子图数组, range(0, 360, 60)):
    子图.imshow(反归一化(生成弧线图片(角度)[0]))
    子图.axis('off')
    子图.set_title(f'{角度}度', fontsize=9)

plt.suptitle('合成弧线刺激（6 个方向，每隔 60 度）', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def 测量方向调谐(模型, 层名, 通道, 角度数=36):
    """把弧线旋转一圈（0°~360°），测量每个角度下该神经元的激活值。

    返回：
        角度数组: 所有测试角度（numpy 数组）
        激活值数组: 对应的激活强度（numpy 数组）
    """
    角度数组 = np.linspace(0, 360, 角度数, endpoint=False)  # 均匀分布的 36 个角度
    激活值列表 = []
    暂存 = {}

    # Hook 记录激活值（空间维度取均值，得到一个标量）
    def 记录(m, 输入, 输出):
        暂存['值'] = 输出[0, 通道].mean().item()

    钩子 = dict(模型.named_modules())[层名].register_forward_hook(记录)

    with torch.no_grad():
        for 角度 in 角度数组:
            模型(生成弧线图片(角度).to(device))
            激活值列表.append(暂存['值'])

    钩子.remove()
    return 角度数组, np.array(激活值列表)


# 对两个曲线探测器神经元分别测量
待测神经元 = [('mixed3b', 379), ('mixed3b', 425)]
print(f'测量方向调谐（每个神经元转 36 个角度，共 {36*len(待测神经元)} 次前向传播）...')

调谐结果 = []
for 层名, 通道 in 待测神经元:
    角度数组, 激活值数组 = 测量方向调谐(model, 层名, 通道)
    调谐结果.append((角度数组, 激活值数组, f'{层名}:{通道}'))
    偏好方向 = 角度数组[激活值数组.argmax()]
    print(f'  {层名}:{通道} 的偏好方向 = {偏好方向:.0f}°')

In [ ]:
图, 子图数组 = plt.subplots(
    1, len(调谐结果),
    figsize=(5*len(调谐结果), 4.5),
    subplot_kw={'projection': 'polar'}
)
if len(调谐结果) == 1:
    子图数组 = [子图数组]

for 子图, (角度数组, 激活值数组, 标签) in zip(子图数组, 调谐结果):
    归一化激活 = (激活值数组 - 激活值数组.min()) / (激活值数组.max() - 激活值数组.min() + 1e-8)
    弧度数组 = np.append(np.deg2rad(角度数组), np.deg2rad(角度数组[0]))
    值数组 = np.append(归一化激活, 归一化激活[0])
    子图.plot(弧度数组, 值数组, linewidth=2, color='steelblue')
    子图.fill(弧度数组, 值数组, alpha=0.25, color='steelblue')
    子图.set_title(标签, pad=15)
    子图.set_theta_zero_location('N')
    子图.set_theta_direction(-1)

plt.suptitle('方向调谐（极坐标图）\n尖峰方向 = 偏好方向，半径 = 归一化激活值', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('结论：曲线探测器对特定方向的弧线响应显著更强，与视觉皮层 V1 简单细胞行为吻合。')

---
## 第 5 节：电路分析——曲线是如何被「计算」出来的？

**电路主张**：曲线探测器不是凭空检测曲线的，
它是通过**读取上一层边缘探测器的输出**，再加权组合而成的。

```
mixed3a 的边缘探测器  ──  权重矩阵  ──→  mixed3b 的曲线探测器
（能检测各方向的边缘）                   （能检测特定方向的曲线）
```

我们通过直接读取权重矩阵来验证这个电路：
找出对目标曲线探测器影响最大的上游神经元，然后可视化它们。

### mixed3b:379 属于哪个分支？
- 通道 379 落在 5×5 分支（范围 320~415）
- 分支内索引 = 379 - 320 = **59**
- 5×5 分支结构：`mixed3a(256ch)` → `1×1压缩(32ch)` → `5×5卷积(96ch)`

In [ ]:
# ── 读取权重，分析 mixed3b:379 的上游依赖 ──────────────────

目标曲线通道 = 379
分支内索引 = 目标曲线通道 - 320   # = 59

# 获取 5×5 分支的两个权重矩阵
# mixed3b_5x5_bottleneck: 1×1 压缩卷积，把 mixed3a 的 256 个通道压缩到 32 个
# mixed3b_5x5:            5×5 卷积，把 32 个通道扩展到 96 个
权重_压缩 = model.mixed3b_5x5_bottleneck_pre_relu_conv.weight.data.cpu()  # 形状 (32, 256, 1, 1)
权重_5x5  = model.mixed3b_5x5_pre_relu_conv.weight.data.cpu()             # 形状 (96, 32, 5, 5)

print(f'1×1 压缩层权重形状: {tuple(权重_压缩.shape)}')
print(f'  含义：32个输出通道，每个通道连接 mixed3a 的全部 256 个通道')
print()
print(f'5×5 卷积层权重形状: {tuple(权重_5x5.shape)}')
print(f'  含义：96个输出通道，每个通道的 5×5 滤波器作用于 32 个压缩通道')

# 步骤1：目标通道（分支内索引59）的 5×5 滤波器
目标滤波器 = 权重_5x5[分支内索引]           # 形状 (32, 5, 5)
# 对空间维度(5×5)取绝对值求和 → 每个压缩通道对目标的「总影响力」
压缩通道重要性 = 目标滤波器.abs().sum((1, 2))  # 形状 (32,)

# 步骤2：通过压缩层权重，把「压缩通道重要性」映射回 mixed3a 的 256 个通道
压缩层权重矩阵 = 权重_压缩.squeeze()          # 形状 (32, 256) — 去掉多余的 1×1 维度
# 加权求和：每个 mixed3a 通道的综合重要性
mixed3a通道重要性 = (压缩通道重要性[:, None] * 压缩层权重矩阵.abs()).sum(0)  # 形状 (256,)

# 找出影响力最大的前 8 个 mixed3a 上游通道
K = 8
前K重要通道 = mixed3a通道重要性.topk(K)
上游通道索引 = 前K重要通道.indices.tolist()
上游重要性值 = 前K重要通道.values.tolist()

print()
print(f'对 mixed3b:{目标曲线通道}（曲线探测器）影响最大的 {K} 个上游神经元：')
for 排名, (索引, 重要性) in enumerate(zip(上游通道索引, 上游重要性值)):
    print(f'  第{排名+1}名: mixed3a:{索引:3d}  综合影响力 = {重要性:.3f}')

# ── 可视化全部 256 个上游通道的重要性分布 ──
plt.figure(figsize=(12, 3))
排序索引 = mixed3a通道重要性.argsort(descending=True)
排序值 = mixed3a通道重要性[排序索引].numpy()
颜色 = ['steelblue' if i < K else 'lightgray' for i in range(len(排序值))]
plt.bar(range(len(排序值)), 排序值, color=颜色, width=1.0)
plt.xlabel('mixed3a 通道（按重要性排序）')
plt.ylabel('综合影响力')
plt.title(f'mixed3a 所有 256 个通道对 mixed3b:{目标曲线通道} 的影响力分布\n蓝色 = 前 {K} 名', fontsize=10)
plt.tight_layout()
plt.show()
print('观察：少数通道有很高的影响力，大多数通道的贡献接近于零 → 稀疏连接')

In [ ]:
取前几个 = 4

print(f'正在生成前 {取前几个} 个上游神经元的特征可视化图（约 2 分钟）...')
上游图片列表 = []
for 通道 in 上游通道索引[:取前几个]:
    print(f'  mixed3a:{通道}', end=' ... ', flush=True)
    图片 = 可视化神经元('mixed3a', 通道, 步数=默认步数, 显示=False)
    上游图片列表.append((通道, 图片))
    print('完成')

曲线探测器图片, _ = 缓存[('mixed3b', 目标曲线通道)]

# 使用 GridSpec 让箭头列更窄，避免大面积空白
宽度比例 = [1]*取前几个 + [0.4] + [1]  # 箭头列只占 0.4 倍宽度
图 = plt.figure(figsize=((取前几个+1.4)*3.2, 3.5))
网格 = gridspec.GridSpec(1, 取前几个+2, width_ratios=宽度比例, wspace=0.15)

for i, (通道, 图片) in enumerate(上游图片列表):
    子图 = 图.add_subplot(网格[0, i])
    子图.imshow(图片)
    子图.axis('off')
    子图.set_title(f'上游 #{i+1}\nmixed3a:{通道}\n（边缘探测器）', fontsize=8.5)

箭头图 = 图.add_subplot(网格[0, 取前几个])
箭头图.text(0.5, 0.5, '加权\n求和\n\u279c',
    ha='center', va='center', fontsize=13, color='gray',
    transform=箭头图.transAxes)
箭头图.axis('off')

目标图 = 图.add_subplot(网格[0, 取前几个+1])
目标图.imshow(曲线探测器图片)
目标图.axis('off')
目标图.set_title(f'目标：曲线探测器\nmixed3b:{目标曲线通道}', fontsize=8.5)

plt.suptitle('电路：边缘探测器(mixed3a) \u279c 曲线探测器(mixed3b)', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('这就是一个完整的「电路」：上游边缘探测器 -> 加权组合 -> 曲线探测器')

---
## 第 6 节：普遍性验证

**普遍性主张**：边缘探测器、曲线探测器这些特征不是 InceptionV1 独有的，
在其他架构的模型中也会自发出现。

我们用 **ResNet-18** 来验证：
- 架构完全不同（残差网络 vs Inception 模块）
- 独立训练（不同的随机种子、可能不同的训练流程）
- 但同样在 ImageNet 上训练

In [ ]:
resnet模型 = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device).eval()
print('ResNet-18 加载完成')

第一层权重 = resnet模型.conv1.weight.data.cpu()  # (64, 3, 7, 7)

图, 子图数组 = plt.subplots(4, 8, figsize=(12, 6))
for i, 子图 in enumerate(子图数组.flatten()):
    滤波器 = 第一层权重[i].permute(1, 2, 0).numpy()
    滤波器 = (滤波器 - 滤波器.min()) / (滤波器.max() - 滤波器.min() + 1e-8)
    子图.imshow(滤波器)
    子图.axis('off')
    子图.set_title(str(i), fontsize=6)

plt.suptitle('ResNet-18 第一层滤波器（64 个 7x7 滤波器，直接可视化权重）', fontsize=10)
plt.tight_layout()
plt.show()
print('注意：能看到边缘/颜色/方向等滤波器，与 InceptionV1 早期层类似')

In [ ]:
# 方法2：对 ResNet-18 做激活最大化（与 InceptionV1 完全一样的方法）
# lucent 的 render_vis 支持任意 PyTorch 模型

resnet待可视化神经元 = [
    # (层名,    通道,  描述)
    # layer1 对应 InceptionV1 的 mixed3a 层级（较浅）
    ('layer1',  0,  'ResNet layer1:0'),
    ('layer1',  8,  'ResNet layer1:8'),
    ('layer1', 16,  'ResNet layer1:16'),
    # layer2 对应更深一层
    ('layer2',  0,  'ResNet layer2:0'),
    ('layer3',  0,  'ResNet layer3:0'),
]

print(f'对 ResNet-18 做特征可视化（{len(resnet待可视化神经元)} 个神经元）...')
resnet可视化结果 = []
for 层名, 通道, 标签 in resnet待可视化神经元:
    print(f'  {标签}', end=' ... ', flush=True)
    目标 = objectives.channel(层名, 通道)
    结果 = render.render_vis(
        resnet模型, 目标,
        thresholds=(默认步数,),
        show_inline=False,
        progress=False
    )
    resnet可视化结果.append((结果[0][0], 标签))
    print('完成')
print('全部完成')

In [ ]:
inception对比图 = [
    (缓存[('conv2d0',   0)][0],  'InceptionV1\nconv2d0:0\n（早期特征）'),
    (缓存[('mixed3a',   0)][0],  'InceptionV1\nmixed3a:0\n（边缘探测器）'),
    (缓存[('mixed3b', 379)][0],  'InceptionV1\nmixed3b:379\n（曲线探测器）'),
    (缓存[('mixed3b',   6)][0],  'InceptionV1\nmixed3b:6\n（高低频）'),
]

列数 = max(len(inception对比图), len(resnet可视化结果))
图, 子图数组 = plt.subplots(2, 列数, figsize=(列数*3, 6.5))

for i, (图片, 标签) in enumerate(inception对比图):
    子图数组[0, i].imshow(图片)
    子图数组[0, i].axis('off')
    子图数组[0, i].set_title(标签, fontsize=8)
for j in range(len(inception对比图), 列数):
    子图数组[0, j].axis('off')

for i, (图片, 标签) in enumerate(resnet可视化结果):
    子图数组[1, i].imshow(图片)
    子图数组[1, i].axis('off')
    子图数组[1, i].set_title(标签, fontsize=8)
for j in range(len(resnet可视化结果), 列数):
    子图数组[1, j].axis('off')

plt.suptitle('普遍性验证：InceptionV1（上）vs ResNet-18（下）\n不同架构、独立训练，却涌现出相似的早期特征', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('结论：两个完全不同架构、独立训练的模型，早期层都出现了相似的边缘/曲线特征。')
print()
print('注意：精确的神经元一一对应是困难的，因为：')
print('  1. 不同模型的通道排列顺序不同')
print('  2. 存在多义性（polysemanticity）：一个神经元可能同时响应多种不相关的特征')
print('  3. 普遍性更多体现在「特征类型」层面，而非具体通道编号')

---
## 第 7 节：局限性与开放问题

本教程展示了 Circuits 的核心思想，但也有重要的局限性值得讨论。

In [ ]:
print('=' * 60)
print('局限性与开放问题')
print('=' * 60)
print()
print('1. 数据集分辨率')
print('   本教程使用 CIFAR-10（32×32 放大到 224×224），分辨率较低。')
print('   使用 ImageNet 子集或更高分辨率的数据集会得到更清晰的验证结果。')
print()
print('2. 多义性（Polysemanticity）')
print('   本教程假设每个神经元对应一个清晰的特征，但实际上')
print('   许多神经元会同时响应多种不相关的特征（多义性）。')
print('   这是 Toy Models of Superposition（Elhage et al., 2022）的核心研究问题。')
print()
print('3. 电路分析的简化')
print('   第 5 节用权重绝对值作为"影响力"的代理指标，')
print('   但这忽略了 ReLU 非线性、特征交互、以及输入分布的影响。')
print('   更严格的方法包括 activation patching 和 causal scrubbing。')
print()
print('4. 从视觉到语言')
print('   本教程聚焦于视觉模型（CNN），但当前可解释性研究的前沿')
print('   已转向 Transformer 语言模型。关键概念包括：')
print('   - Induction Heads（归纳头）：Transformer 中的经典电路')
print('   - Sparse Autoencoders（SAE）：解决多义性的新方法')
print('   - TransformerLens：分析 GPT 风格模型的工具库')
print()
print('这些局限性恰恰指向了可解释性研究中最活跃的方向。')

---
## 总结

| 主张 | 本 Notebook 的实验 | 核心结论 |
|------|-------------------|----------|
| **特征** | 第2节：激活最大化<br>第3节：数据集样本验证 | 神经元检测可理解的视觉模式，且与真实图片一致 |
| **电路** | 第4节：方向调谐<br>第5节：权重分析 | 曲线探测器 = 加权组合上游边缘探测器 |
| **普遍性** | 第6节：跨架构对比 | 不同模型独立训练后出现相似的基础特征 |
| **局限性** | 第7节：开放问题 | 多义性、非线性交互、向 Transformer 的延伸 |

---
## 动手探索（跑完后来这里玩）

```python
# 1. 随便看一个你感兴趣的神经元
可视化神经元('mixed4e', 100)
可视化神经元('mixed5b', 0)

# 2. 扫描一层，看看都有哪些特征
for ch in range(0, 480, 48):
    可视化神经元('mixed3b', ch, 标题=f'mixed3b:{ch}')

# 3. 测试一个你自己找到的神经元的方向调谐
角度数组, 激活值数组 = 测量方向调谐(model, 'mixed3b', 200)
plt.polar(np.deg2rad(角度数组), 激活值数组)
plt.show()
```

---
## 延伸阅读

- [Zoom In: An Introduction to Circuits](https://distill.pub/2020/circuits/zoom-in/) — 本教程复现的原论文
- [Curve Detectors](https://distill.pub/2020/circuits/curve-detectors/) — 曲线探测器的深入分析
- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) — 特征叠加理论
- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) — Transformer 电路的数学框架
- [TransformerLens](https://github.com/neelnanda-io/TransformerLens) — 分析 GPT 风格模型的 Python 工具库